## Parametrization and Channel Generation

In [ ]:
import sys; sys.path.append('../..')
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf
from tri_mesh_viewer import TriMeshViewer

target_surf = mesh.Mesh('../../SiggraphExamples/Meshes/20200118_bike_helmet_v1_R00.obj')
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
# Choose reasonable stretching bounds
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))
print(alphaMin, alphaMax)

In [ ]:
lg = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

for i in range(1000): lg.runIteration()
print(lg.energy())
lg.alphaMin = alphaMin
lg.alphaMax = alphaMax

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [ ]:
for i in range(1000): lg.runIteration()
print(lg.energy())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(lg)

In [ ]:
for i in range(5000): lg.runIteration()
print(lg.energy())

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surf, lg.uv())
rparam.alphaMin = alphaMin
rparam.alphaMax = alphaMax

In [ ]:
opts = NewtonOptimizerOptions()

In [ ]:
def optimize_rparam(param, alphaRegW, phiRegW, bendRegW):
    param.alphaRegW = alphaRegW
    param.phiRegW = phiRegW
    param.bendRegW = bendRegW
    opts = NewtonOptimizerOptions()
    opts.niter = 1000
    opts.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
    #opts.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()
    cr = parametrization.regularized_parametrization_newton(param, param.rigidMotionPinVars, opts)

In [ ]:
# Rerun this cell until convergence (twice)
with suppress_stdout(): optimize_rparam(rparam, 5e2, 1e1, 500.0)
with suppress_stdout(): optimize_rparam(rparam, 1e2, 1e0, 500.0)
with suppress_stdout(): optimize_rparam(rparam, 1e2, 1e-1, 500.0)

with suppress_stdout(): optimize_rparam(rparam, 1e1, 1e-2, 250.0)
with suppress_stdout(): optimize_rparam(rparam, 1e0, 1e-2, 250.0)
with suppress_stdout(): optimize_rparam(rparam, 1e-1, 5e-3, 100.0)
with suppress_stdout(): optimize_rparam(rparam, 1e-1, 5e-3, 25.0)
with suppress_stdout(): optimize_rparam(rparam, 1e-2, 2.5e-3, 12.5)

In [ ]:
utils.allGradientNorms(rparam), utils.allEnergies(rparam)

In [ ]:
visualization.visualize(rparam)

In [ ]:
importlib.reload(visualization)
visualization.visualizeChannelOrientation(rparam, quiver=visualization.QuiverVisualization.PER_VTX, orientationHue=False)

In [ ]:
visualization.singularValueHistogram(rparam)

In [ ]:
visualization.singularValueHistogram(rparam)

In [ ]:
utils.save(rparam.uv(), 'data/helmet_uv_2.pkl.gz')

In [ ]:
widths = wwf.wallWidthForCanonicalWidth(wwf.canonicalWallWidthForStretchFactor(rparam.getAlphas()), 10)
(np.min(widths), np.max(widths))